# Clase 9 · El parte de incendios (gemelo, trabajo autónomo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este proyecto lo hacen por su cuenta, fuera del salón. Es el gemelo de *El noticiero de un minuto*: un análisis que termina convertido en un servicio. Tiene dos partes. En este cuaderno justifican con números a qué departamentos mandarían las brigadas. En la carpeta `parte_de_incendios/` construyen la API que entrega el parte.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Construir, probar y desplegar una API sobre datos reales, trabajando con una IA que escribe el código contra pruebas de aceptación que ustedes no escribieron. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular), 2 (preparar datos), 3 (flujo reproducible), 4 (implementar y evaluar), 5 (comunicar) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | Claude escribe la API y, si quieren, el análisis. Ustedes deben poder explicar sin ayuda qué cuenta cada ruta, por qué `/brigadas` solo puede mirar el pasado, cómo midieron su método y qué hace cada código de respuesta. |
| **Entregable** | La dirección de su repositorio, la dirección pública de su API en Railway y este cuaderno corrido, con la sección 3 llena. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

In [ ]:
# El parámetro del cuaderno. Al final se cambia esta línea y se corre todo de nuevo.
N_BRIGADAS = 5            # a cuántos departamentos alcanza a ir la entidad en una semana

import time

## 1. El enunciado

Imagínense la sala de crisis de una entidad de gestión del riesgo. Cada lunes alguien pregunta cuántos focos de calor hubo en un departamento y a dónde conviene mandar las brigadas esta semana. Hay brigadas para `N_BRIGADAS` departamentos.

1. **Predecir.** ¿Cuántos focos de calor habrá la próxima semana en cada departamento?
2. **Explicar.** ¿Qué pesa más en esa cifra, la época del año o lo que pasó la semana anterior?
3. **Decidir.** ¿A qué departamentos mandas las brigadas la próxima semana?

**La métrica.** Una propuesta de brigadas es un conjunto $B_t$ de departamentos para la semana $t$. Si $y_{d,t}$ son los focos que de verdad hubo en el departamento $d$, la **cobertura** de la propuesta es la fracción de los focos del país que cayó donde había brigada:

$$\text{cobertura}_t=\frac{\sum_{d\in B_t}y_{d,t}}{\sum_{d}y_{d,t}}.$$

Nadie puede cubrirlo todo con cinco brigadas. El **techo** es la cobertura de quien conociera el futuro y escogiera los departamentos que de verdad tuvieron más focos, $B_t^\ast$. Un método se juzga por cuánto se acerca al techo y por cuánto le gana a la línea base, que es mandar las brigadas a donde hubo más focos la semana pasada.

### Criterios de aceptación

**De la API** (carpeta `clases/09_api/parte_de_incendios/`, léanlos completos en su `README.md`): la entrega se acepta cuando `python pruebas_aceptacion.py` termina con «10 de 10 criterios cumplidos». Las diez pruebas ya están escritas y no se cambian.

**De este cuaderno:**

1. Corre de principio a fin desde cero.
2. Muestra la exploración del dato: focos por año, por mes y por departamento, con su lectura.
3. Compara al menos tres métodos para `/brigadas` contra la línea base, con tramos de entrenamiento, validación y prueba **separados en el tiempo**, y reporta la cobertura y el techo.
4. Muestra con un número qué pasa cuando el método usa datos del futuro.
5. Termina con la propuesta para una semana concreta, su incertidumbre y la decisión en tres frases.

### Predicción escrita

- ¿En qué meses se concentran los focos de calor en Colombia? **Mi apuesta:**
- Con cinco brigadas y conociendo el futuro, ¿qué fracción de los focos de una semana se alcanza a cubrir? **Mi apuesta:**
- ¿Qué servirá más para acertar: saber qué pasó la semana pasada o saber qué pasa en esta época todos los años? **Mi apuesta:**

## 2. El dato

NASA FIRMS publica los focos de calor que detecta el sensor MODIS de los satélites Terra y Aqua. Cada fila es un píxel de cerca de un kilómetro donde el satélite vio una anomalía térmica. La función trae los archivos anuales de 2016 a 2025 y le agrega a cada foco su departamento, calculado con los límites de geoBoundaries. Tarda unos veinte segundos.

Según la documentación de FIRMS, `confidence` va de 0 a 100, `frp` es la potencia radiativa del fuego en megavatios, `acq_time` es la hora de paso del satélite en tiempo universal y `type` vale 0 para presunto incendio de vegetación, 1 para volcán activo, 2 para otra fuente fija en tierra y 3 para mar adentro.

In [ ]:
focos = descargar("firms_colombia")
focos["fecha"] = pd.to_datetime(focos["acq_date"])
print("Focos:", len(focos), "| de", focos["fecha"].min().date(), "a", focos["fecha"].max().date())
print("Por tipo:", focos["type"].value_counts().to_dict(), "| sin departamento:", int(focos["departamento"].isna().sum()))
focos.head(3)

## 3. Su análisis

De aquí en adelante el cuaderno es suyo. Sigan los cuatro módulos del curso: datos y exploración, modelado, y comunicación. Los títulos son una guía.

### 3.1 Limpieza y exploración

In [ ]:
# Limpieza (qué focos entran y por qué) y exploración: por año, por mes, por departamento, en el mapa.

**Mi lectura** (¿cuándo y dónde se concentran los focos?, ¿qué decisión de limpieza tomaron y por qué?):

### 3.2 Un análisis no supervisado: ¿todos los departamentos tienen la misma temporada?

El perfil estacional del departamento $d$ es el vector de doce números $s_d=(s_{d,1},\dots,s_{d,12})$, con $s_{d,m}$ igual a la fracción de sus focos que cae en el mes $m$. Un PCA sobre esos perfiles dice si hay una sola temporada nacional o varias.

In [ ]:
# Perfil estacional por departamento (fracción de focos en cada mes), PCA de los perfiles y su lectura.

**Mi lectura** (¿hay una sola temporada de incendios en el país?):

### 3.3 Modelado: la tabla semanal, la línea base y los candidatos

La unidad es la semana. Sea $y_{d,t}$ el número de focos del departamento $d$ en la semana $t$. Todos los candidatos producen un valor esperado $\hat y_{d,t}$ usando solo semanas anteriores a $t$, y la propuesta de brigadas son los `N_BRIGADAS` departamentos con mayor $\hat y_{d,t}$.

| Candidato | $\hat y_{d,t}$ |
|---|---|
| Línea base: la semana pasada | $y_{d,t-1}$ |
| Promedio de cuatro semanas | $\tfrac14\sum_{k=1}^{4}y_{d,t-k}$ |
| Estacional | $c_{d,t}$: promedio de $y_d$ en la misma semana del año, y sus dos vecinas, en los años **anteriores** |
| Estacional con nivel | $c_{d,t}\cdot\rho_{d,t}$, con $\rho_{d,t}=\dfrac{1+\sum_{k=1}^{4}y_{d,t-k}}{1+\sum_{k=1}^{4}c_{d,t-k}}$ acotado entre $\tfrac14$ y 4 |
| Regresión de Poisson | $\log\hat y_{d,t}=\beta_0+\beta^\top x_{d,t}$, con $x$ hecho de rezagos, estacional y época del año |
| *Gradient boosting* de Poisson | Árboles sobre las mismas entradas |

Los tramos van en el tiempo: los modelos se ajustan con 2018 a 2022, se escoge con 2023 y se reporta con 2024 y 2025. Los años 2016 y 2017 solo sirven para que el estacional tenga pasado.

In [ ]:
# Tabla semanal de focos por departamento, la métrica de cobertura, la línea base y los candidatos sin modelo.

In [ ]:
# Dos modelos ajustados (por ejemplo, regresión de Poisson y boosting), la tabla comparativa con el techo,
# el ganador escogido EN VALIDACIÓN y su ventaja sobre la línea base en prueba, con intervalo.

**Mi lectura** (¿quién gana?, ¿la ventaja se distingue del ruido?, ¿qué tan lejos queda del techo?):

### 3.4 La trampa: mirar el futuro

Hay dos descuidos que dejan entrar el futuro, y los dos salen de escribir una línea de pandas sin pensar en el calendario.

- **El promedio móvil sin correr.** `rolling(4).mean()` promedia las cuatro semanas que terminan en $t$, incluida la semana $t$, que es la que se quiere adivinar. La versión correcta corre la tabla una semana antes de promediar.
- **El estacional con todos los años.** Si el promedio de «la misma época» se calcula una sola vez con toda la tabla, para pronosticar 2023 se están usando 2024 y 2025.

Midan cada descuido contra su versión correcta.

In [ ]:
# Los dos descuidos, cada uno contra su versión correcta: el promedio móvil que incluye la semana pronosticada
# y el estacional calculado con años futuros. Mostrar la cobertura de las cuatro versiones y las diferencias.

**Mi explicación** (¿cuál de los dos descuidos infla más y por qué?, ¿qué cifras puede cumplir la API un lunes cualquiera?, ¿qué criterio de aceptación protege contra esta trampa?). Ojo con el segundo: que un descuido casi no cambie el número no lo vuelve correcto, solo dice que la temporada se repite mucho de un año a otro.

### 3.5 Explicar: ¿la época del año o la semana pasada?

In [ ]:
# Una respuesta con números a la pregunta de explicar: por ejemplo, los coeficientes de la regresión de Poisson
# y la cobertura de un modelo que solo ve rezagos contra uno que solo ve la época del año.

**Mi lectura:**

### 3.6 Decidir: la propuesta de una semana y su incertidumbre

Usen el método más simple entre los que no pierden: si el ganador de validación no le saca una ventaja clara al estacional, la API se queda con el estacional, que cabe en una frase. Den la propuesta para la semana siguiente a la última completa y digan cuánto puede fallar. La cobertura cambia de una semana a otra, y cambia entre la temporada seca y el resto del año.

In [ ]:
# La distribución de la cobertura semanal del método elegido en la prueba (media, percentiles 10 y 90,
# temporada seca contra el resto) y la propuesta de departamentos para la semana siguiente al último dato.

### La decisión en tres frases

Para quien dirige la sala de crisis.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 3.7 La API

Corran aquí las pruebas de aceptación de su API y dejen el resultado a la vista.

In [ ]:
# Corran `python pruebas_aceptacion.py` dentro de la carpeta de su API y muestren la salida.
# Por ejemplo:  import subprocess; print(subprocess.run(["python", "pruebas_aceptacion.py"], cwd="RUTA/A/SU/CARPETA", capture_output=True, text=True).stdout)

Dirección pública de mi API: 

Dirección de mi repositorio: 

### 3.8 Bitácora, riesgos y variante

**Bitácora de decisiones.** Al menos cinco filas.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| | | |

**Riesgos y mantenimiento.** Respondan en dos o tres líneas cada una.

- Un foco no es un incendio, y bajo nubes el satélite no ve. ¿Qué error comete quien lee «cero focos» como «cero incendios»?
- Los departamentos con más focos son siempre los mismos. Si las brigadas van siempre allá, ¿quién queda sin atención? ¿Qué otra métrica propondrían para que la decisión no dependa solo del conteo?
- La fecha del archivo es universal y los límites vienen de OpenStreetMap. ¿En qué casos cambia eso el parte de un departamento?
- Su API calcula `/brigadas` con datos hasta 2025. ¿Qué habría que hacer para que sirva un lunes de 2027, y cómo se enteraría alguien de que los datos están viejos?

**Cambien una línea.** Pongan `N_BRIGADAS = 3`, corran todo y llenen la tabla.

| | 5 brigadas | 3 brigadas |
|---|---|---|
| Techo en prueba | | |
| Ganador en validación | | |
| Cobertura del ganador en prueba | | |
| Cobertura de la línea base en prueba | | |

¿Cuánto cuesta, en focos sin cubrir, tener dos brigadas menos? **Mi respuesta:**

### Procedencia y entrega

In [ ]:
{nombre: ficha for nombre, ficha in procedencia().items() if nombre == "firms_colombia"}

**Para entregar:** el cuaderno corrido en su portafolio, carpeta `clase09/`, con las dos direcciones escritas en la sección 3.7.